# Intent Classifier for RAG Chatbot

Fine-tuning **ruBERT-tiny2** for intent classification (rag / chat / followup).

**Goal:** Replace LLM API calls for message classification with a local model.

| Metric | LLM API (before) | Local model (after) |
|--------|-------------------|---------------------|
| Latency | 300-2000ms | ~3ms |
| Cost | ~$0.001/req | $0 |
| Size | 120B params (remote) | 29M params (15MB local) |
| Reliability | Depends on API | Offline, deterministic |

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate scikit-learn onnx onnxruntime onnxscript

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU found. Training will be slower but still work.")

## 2. Load Dataset

Upload `dataset.csv` to Colab (drag & drop into the file panel on the left).

In [ ]:
# If running in Colab, upload the file first
import os
if not os.path.exists("dataset.csv"):
    from google.colab import files
    uploaded = files.upload()  # Upload dataset.csv here

In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Load CSV
df = pd.read_csv("dataset.csv")
print(f"Total examples: {len(df)}")
print(f"\nDistribution:")
print(df["label"].value_counts())

# Create label mapping
LABELS = ["rag", "chat", "followup"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for i, label in enumerate(LABELS)}

df["label_id"] = df["label"].map(label2id)

# Split: 70% train, 15% validation, 15% test
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print(f"\nSplit sizes:")
print(f"  Train:      {len(train_df)}")
print(f"  Validation: {len(val_df)}")
print(f"  Test:       {len(test_df)}")

# Convert to HuggingFace Dataset
def df_to_dataset(dataframe):
    return Dataset.from_dict({
        "text": dataframe["text"].tolist(),
        "label": dataframe["label_id"].tolist(),
    })

dataset = DatasetDict({
    "train": df_to_dataset(train_df),
    "validation": df_to_dataset(val_df),
    "test": df_to_dataset(test_df),
})

print(f"\nDataset ready:")
print(dataset)

## 3. Tokenization

Convert text to numbers that the model understands.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "cointegrated/rubert-tiny2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128,  # More than enough for short messages
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Show example
example = tokenized_dataset["train"][0]
print(f"Text: {dataset['train'][0]['text']}")
print(f"Label: {id2label[example['label']]}")
print(f"Tokens: {example['input_ids'][:20]}...")
print(f"Token count: {sum(1 for t in example['input_ids'] if t != 0)}")

## 4. Model Setup

Load ruBERT-tiny2 and add a classification head (3 outputs: rag, chat, followup).

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

# Model size info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,} ({total_params / 1e6:.1f}M)")
print(f"Trainable: {trainable_params:,}")
print(f"Estimated model size: {total_params * 4 / 1e6:.1f} MB (float32)")

## 5. Training

Fine-tune the model. This should take ~5-10 minutes on a T4 GPU.

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    """Calculate accuracy, precision, recall, F1 for each class."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


training_args = TrainingArguments(
    output_dir="./results",
    
    # Training
    num_train_epochs=5,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    
    # Logging
    logging_steps=50,
    report_to="none",  # No W&B or other tracking
    
    # Misc
    seed=42,
    fp16=torch.cuda.is_available(),  # Mixed precision on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

print("Starting training...")
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"Training time: {train_result.metrics['train_runtime']:.0f}s")

## 6. Evaluation

Test the model on data it has never seen.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Evaluate on test set
predictions = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Classification report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(
    true_labels, preds,
    target_names=LABELS,
    digits=3,
))

# Confusion matrix
cm = confusion_matrix(true_labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=LABELS, yticklabels=LABELS,
)
plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print(f"\nOverall accuracy: {accuracy_score(true_labels, preds):.1%}")

## 7. Interactive Testing

Try the model with custom messages.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

# Test messages
test_messages = [
    # RAG
    "какие условия возврата товара?",
    "найди в документе информацию о сроках",
    "сколько стоит годовая подписка?",
    "какой штраф за просрочку?",
    
    # Chat
    "привет!",
    "что ты умеешь?",
    "спасибо за помощь",
    "как загрузить документ?",
    
    # Followup
    "расскажи подробнее",
    "а почему именно так?",
    "можно пример?",
    "объясни проще",
    
    # Edge cases
    "а?",
    "хм",
    "ну расскажи про доставку подробнее",
    "а если клиент отказывается?",
]

print(f"{'Message':<45} {'Predicted':>10} {'Confidence':>10}")
print("-" * 70)
for msg in test_messages:
    result = classifier(msg)[0]
    label = result["label"]
    score = result["score"]
    print(f"{msg:<45} {label:>10} {score:>9.1%}")

## 8. Export to ONNX

Convert model to ONNX format for lightweight production deployment (no PyTorch needed).

In [ ]:
import torch
import os
from pathlib import Path
import onnxruntime
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_DIR = "./onnx_model"
QUANTIZED_DIR = "./onnx_model_quantized"
os.makedirs(ONNX_DIR, exist_ok=True)
os.makedirs(QUANTIZED_DIR, exist_ok=True)

# Step 1: Save PyTorch model
PYTORCH_DIR = "./pytorch_model"
model.save_pretrained(PYTORCH_DIR)
tokenizer.save_pretrained(PYTORCH_DIR)

# Step 2: Export to ONNX via torch.onnx
model.eval()
model.cpu()
dummy = tokenizer("test", return_tensors="pt", padding="max_length", truncation=True, max_length=128)
onnx_path = os.path.join(ONNX_DIR, "model.onnx")

torch.onnx.export(
    model,
    (dummy["input_ids"], dummy["attention_mask"]),
    onnx_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch"}, "attention_mask": {0: "batch"}},
    opset_version=14,
)
tokenizer.save_pretrained(ONNX_DIR)

# Step 3: Quantize (INT8)
quantized_path = os.path.join(QUANTIZED_DIR, "model_quantized.onnx")
quantize_dynamic(onnx_path, quantized_path, weight_type=QuantType.QUInt8)
tokenizer.save_pretrained(QUANTIZED_DIR)

# Check sizes
def dir_size_mb(path):
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return total / 1e6

print(f"\nModel sizes:")
print(f"  PyTorch:        {dir_size_mb(PYTORCH_DIR):.1f} MB")
print(f"  ONNX:           {dir_size_mb(ONNX_DIR):.1f} MB")
print(f"  ONNX quantized: {dir_size_mb(QUANTIZED_DIR):.1f} MB")

## 9. Test ONNX Model

Verify the ONNX model produces the same results and measure speed.

In [ ]:
import time
import onnxruntime as ort

# Load quantized ONNX model
onnx_path = list(Path(QUANTIZED_DIR).glob("*.onnx"))[0]
session = ort.InferenceSession(str(onnx_path))

def classify_onnx(text: str) -> tuple[str, float]:
    """Classify text using ONNX model. Returns (label, confidence)."""
    inputs = tokenizer(
        text, return_tensors="np",
        padding="max_length", truncation=True, max_length=128,
    )
    outputs = session.run(None, {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
    })
    logits = outputs[0][0]
    probs = np.exp(logits) / np.exp(logits).sum()  # softmax
    pred_id = np.argmax(probs)
    return LABELS[pred_id], float(probs[pred_id])


# Test same messages
print(f"{'Message':<45} {'Label':>10} {'Conf':>6}")
print("-" * 65)
for msg in test_messages:
    label, conf = classify_onnx(msg)
    print(f"{msg:<45} {label:>10} {conf:>5.1%}")

# Benchmark speed
print("\n--- Speed Benchmark ---")
warmup_msg = "какие условия доставки?"
# Warmup
for _ in range(10):
    classify_onnx(warmup_msg)

# Measure
times = []
for _ in range(100):
    start = time.perf_counter()
    classify_onnx(warmup_msg)
    times.append((time.perf_counter() - start) * 1000)

print(f"Average: {np.mean(times):.1f} ms")
print(f"Median:  {np.median(times):.1f} ms")
print(f"P95:     {np.percentile(times, 95):.1f} ms")
print(f"P99:     {np.percentile(times, 99):.1f} ms")

## 10. Download Results

Download the trained model files.

In [ ]:
import shutil

# Create archive with quantized ONNX model
shutil.make_archive("intent-classifier-onnx", "zip", QUANTIZED_DIR)
print(f"Archive size: {os.path.getsize('intent-classifier-onnx.zip') / 1e6:.1f} MB")

# Also save confusion matrix
print("\nFiles to download:")
print("  1. intent-classifier-onnx.zip — quantized ONNX model")
print("  2. confusion_matrix.png — evaluation results")

# Download in Colab
try:
    from google.colab import files
    files.download("intent-classifier-onnx.zip")
    files.download("confusion_matrix.png")
except ImportError:
    print("Not in Colab — files saved locally.")

## 11. (Optional) Push to HuggingFace Hub

Upload the model to HuggingFace for public access.

In [ ]:
# Uncomment and run to push to HuggingFace Hub
# First, login: run `huggingface-cli login` or use token

# from huggingface_hub import login
# login()  # Enter your HuggingFace token

# HF_REPO = "YOUR_USERNAME/intent-classifier-rubert-tiny2"  # Change this!

# # Push PyTorch model
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)

# # Push ONNX model
# ort_model = ORTModelForSequenceClassification.from_pretrained(QUANTIZED_DIR)
# ort_model.push_to_hub(HF_REPO + "-onnx")
# tokenizer.push_to_hub(HF_REPO + "-onnx")

# print(f"Model pushed to: https://huggingface.co/{HF_REPO}")